In [1]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import pathlib
from datetime import datetime

#install xarray
%pip install xarray
import xarray as xr

Note: you may need to restart the kernel to use updated packages.


In [2]:
Stock_TC = pd.read_excel('Stock_TC.xlsx')
Type_split = pd.read_excel('final_type_split.xlsx')

In [20]:
'''Reindexing to get right shape (Renamed to Type_split_ because of ipynb-issues)'''

#print(Type_split.sum(axis=1))

#Set year as index type split
Type_split_ = Type_split.set_index('Year')

#print(Type_split_.head())

'''Convert to numpy array'''

Type_split_array = Type_split_.to_numpy()
Type_split_array

array([[0.        , 0.        , 0.        , 1.        ],
       [0.        , 0.        , 0.        , 1.        ],
       [0.        , 0.        , 0.        , 1.        ],
       ...,
       [0.17313351, 0.26421469, 0.55415056, 0.00857414],
       [0.17313351, 0.26421469, 0.55415056, 0.00857414],
       [0.17313351, 0.26421469, 0.55415056, 0.00857414]], shape=(501, 4))

In [ ]:
'''Reindexing to get right shape (Renamed to Stock_TC_ because of ipynb-issues)'''
print(type(Stock_TC.values))
print(Stock_TC.values.shape)
Stock_TC_ = Stock_TC.set_index('year')
Stock_TC_.values.shape
print(type(Stock_TC_))

'''Converting Stock_TC_ to numpy array '''
Stock_TC_array = Stock_TC_.to_numpy()
Stock_TC_array

<class 'numpy.ndarray'>
(501, 502)
<class 'pandas.DataFrame'>


array([[1.14643761e+05, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.14643041e+05, 2.00507106e+01, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [1.14642199e+05, 2.00505847e+01, 2.05975434e+01, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [3.94594627e+00, 6.90129372e-04, 7.08950919e-04, ...,
        3.19021875e+04, 0.00000000e+00, 0.00000000e+00],
       [3.94594627e+00, 6.90129372e-04, 7.08950919e-04, ...,
        3.19019872e+04, 3.18824946e+04, 0.00000000e+00],
       [3.94594627e+00, 6.90129372e-04, 7.08950919e-04, ...,
        3.19017528e+04, 3.18822945e+04, 3.18637500e+04]], shape=(501, 501))


# Redo multiplication
Stock_TCJ = Stock_TC.values[:, :, np.newaxis] * Type_split_.values[:, np.newaxis, :]

# Rebuild xarray with labels
Stock_TCJ = xr.DataArray(
    Stock_TCJ,
    dims=["time", "cohort", "type"],
    coords={
        "time":   Stock_TC.index,
        "cohort": Stock_TC.columns,
        "type":   Type_split_.columns,
    }
)




Stock_T = Stock_TCJ.sum(dim=["cohort", "type"]).to_pandas()
Stock_TC_check = Stock_TC.sum(axis=1)

plt.plot(Stock_T, label="From Stock_TCJ", linewidth=0.8)
plt.plot(Stock_TC_check, label="Original Stock_TC", linewidth=0.8, linestyle="--")
plt.legend()
plt.title("Mass conservation check")
plt.grid(False)
plt.show()

In [24]:
Stock_TCJ_np = np.einsum('tc,cj->tcj',Stock_TC_array, Type_split_array)

Stock_TCJ_np.shape

(501, 501, 4)

In [26]:
print(type(Stock_TCJ_np))

<class 'numpy.ndarray'>
